# 01 · 고정 Agent Workflow
Cohere Rerank는 Chat 또는 Work만 분기하고, Work는 Work Planning Agents 오케스트레이터가 전문 Agent 협업을 조율합니다.

**학습 목표:** 공개 코어 API `run_workflow()`를 호출하고, Chat/Work route·오케스트레이션 trace를 관찰합니다.  
**실행 경계:** 모든 Agent는 AI Core LLM만 호출하며 외부 시스템을 변경하지 않습니다.  
**환경 변수:** UI와 코어 workflow 모두 실제 Cohere Rerank 및 AI Core LLM을 사용하므로 루트 `.env` 설정이 필요합니다.

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #2e7d32; background: #e8f5e9; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #2e7d32; letter-spacing: .02em;">🖥️  UI 실행</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">UI: Agent Workflow Chat</h2>
<p style="margin: 7px 0;">Agent graph와 채팅 UI에서 요청을 보냅니다. Cohere Rerank는 Chat 또는 Work만 판별하며, Work는 Work Planning Agents 오케스트레이터가 전문 Agent 협업 단계를 조율해 보여 줍니다.</p>
  </div>
</div>


In [1]:
from pathlib import Path
import sys
project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'shared').exists())
sys.path.insert(0, str(project_root / 'shared'))
sys.path.insert(0, str(project_root / '4_agent_patterns'))

from src.workflow_ui import launch_workflow_ui

workflow_ui = launch_workflow_ui()


* Running on local URL:  http://127.0.0.1:7873
* To create a public link, set `share=True` in `launch()`.


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1a73e8; letter-spacing: .02em;">✅  체크포인트</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">체크포인트</h2>
<ul style="margin: 7px 0 0; padding-left: 1.3em;">
<li>UI에서 Chat/Work 분기와 Work Planning Agents 오케스트레이션 단계를 따라 한 번 대화한다.</li>
<li>UI가 실행되지 않으면 AI Core 환경 설정, Cohere Rerank deployment와 모델 이름을 먼저 점검한다.</li>
<li>아래 코드는 같은 기능의 구성 요소를 자세히 살펴보는 선택 심화 자료다.</li>
</ul>
  </div>
</div>


<hr style="border: 0; border-top: 2px solid #c7d2e1; margin: 32px 0 14px;">
<div style="margin: 0 0 18px; color: #4b5f78; font-size: .86em; font-weight: 700; letter-spacing: .04em;">🧩 아래부터 코드 베이스 실습</div>


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #5f6368; background: #f1f3f4; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #5f6368; letter-spacing: .02em;">🔎  선택 심화</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">선택: 코드로 같은 작업 보기</h2>
<p style="margin: 7px 0;">아래 셀은 UI와 같은 공개 코어 API를 단계별로 호출하는 자료입니다. 기본 실습에서는 실행하지 않아도 됩니다.</p>
  </div>
</div>


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">준비: Chat/Work router와 오케스트레이션</h2>
<p style="margin: 7px 0;">Router는 Cohere Rerank로 Chat 또는 Work만 판별합니다. Work를 선택하면 Work Planning Agents 오케스트레이터가 공통 작업 브리프를 만들고 전문 Agent 협업을 조율합니다.</p>
  </div>
</div>


In [ ]:
# 노트북은 구현을 복사하지 않고 공개 코어 API만 사용합니다.
from pathlib import Path
import sys

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'shared').exists())
sys.path.insert(0, str(project_root / 'shared'))
sys.path.insert(0, str(project_root / '4_agent_patterns'))

from src.food_agents import list_food_agents
from src.workflow import run_workflow


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">1. Food Agent 설정 보기</h2>
<p style="margin: 7px 0;">공개 API가 반환한 6개 Agent의 설명, 실제 System Prompt, Tool 종류와 실행 경계를 확인합니다. Tool은 계산·검증, 읽기 전용 API, Agent-as-a-Tool이며 외부 쓰기는 수행하지 않습니다.</p>
  </div>
</div>


In [ ]:
agent_catalog = list_food_agents()
for agent in agent_catalog:
    print(f"## {agent['agent_label']} ({agent['agent_id']})")
    print(f"설명: {agent['description']}")
    print("\n[System Prompt]")
    print(agent['system_prompt'])
    tools = ', '.join(f"{tool['name']} ({tool['type']})" for tool in agent['tools']) if agent['tools'] else '없음'
    print(f"\n[Tools] {tools}")
    print(f"실행 경계: {agent['execution_boundary']}\n")


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">2. 한 요청의 실행 trace</h2>
<p style="margin: 7px 0;">반환된 Trace는 route 판별과 Agent별 실행·응답 완료 순서를 보여줍니다. 노트북은 이 값을 다시 계산하지 않고 그대로 관찰합니다.</p>
  </div>
</div>


In [ ]:
result = run_workflow("재고를 분석해서 발주가 필요한 품목을 알려줘.")
print(result["route"], "→", result["specialist"])
for event in result["trace"]: print(event)


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">3. 요청별 분기 비교</h2>
<p style="margin: 7px 0;">대화와 업무 요청을 넣어 Cohere Rerank의 Chat/Work 판별을 비교합니다. Work를 선택한 요청은 Work Planning Agents가 전문 Agent 협업을 시작합니다.</p>
  </div>
</div>


In [ ]:
for request in ["안녕하세요. workflow와 loop의 차이를 설명해줘.", "경쟁사 시장 근거를 조사해줘.", "임원용 보고서로 요약해줘.", "재고 조회 앱 UI를 만들어줘."]:
    outcome = run_workflow(request)
    print(f"{outcome['route']:<4} | {str(outcome['specialist']):<14} | {request}")


실험: `src/workflow.py`에서 Work Planning Agents의 공통 브리프가 각 전문 Agent에게 어떻게 전달되는지 확인하세요. Rerank는 Chat/Work 분기만 수행합니다.